In [107]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [110]:
class Performer:
    # Shared across ALL Performer instances
    full_history = []  

    def __init__(self, name: str, role: str, client: any, model: str,
                 system_prompt: str = "", temperature: float = 0.7, max_tokens: int = 500):
        """
        A simple performer in the Yes-And game.
        """
        self.name = name
        self.role = role   # "host" or "player"
        self.client = client
        self.model = model
        self.system_prompt = system_prompt
        self.temperature = temperature
        self.max_tokens = max_tokens

    def set_system_prompt(self, system_prompt: str):
        """Reset or update the system prompt."""
        self.system_prompt = system_prompt

    def _chat(self, extra_user_msg: str = None) -> str:
        """Internal helper to call the model with full history and append response."""
        history = [{"role": "system", "content": self.system_prompt}]
        history.extend(Performer.full_history)
        if extra_user_msg:
            history.append({"role": "user", "content": extra_user_msg})

        resp = self.client.chat.completions.create(
            model=self.model,
            messages=history,
            temperature=self.temperature,
            max_tokens=self.max_tokens
        )
        content = resp.choices[0].message.content

        # Label with performer identity (so others know who said it)
        labeled_content = f"{self.name} says: {content}"

        # Save assistant reply into shared history
        Performer.full_history.append({"role": "assistant", "content": labeled_content})
        return labeled_content

    def start_game(self, user_message: str) -> str:
        """First user message to get things started."""
        Performer.full_history.append({"role": "user", "content": user_message})
        return self._chat()

    def user_interaction(self, user_message: str) -> str:
        """Continue conversation with full history (host + user)."""
        Performer.full_history.append({"role": "user", "content": user_message})
        return self._chat()

    def speak(self) -> str:
        """
        Player speaks by building on the last assistant message in history.
        Example: Wayne uses Ryan's last line as context for his next move.
        """
        # Find the last assistant message (could be host or another player)
        last_line = None
        for msg in reversed(Performer.full_history):
            if msg["role"] == "assistant":
                last_line = msg["content"]
                break

        if last_line is None:
            # If no assistant messages yet, just let the model go
            return self._chat()

        # Add a user prompt to encourage Yes-And on last line
        extra_prompt = f"Continue the scene by building on this: {last_line}"
        return self._chat(extra_user_msg=extra_prompt)

    def decide(self) -> str:
        """
        Host decides if the game should continue or end.
        By default, let the model make the decision based on the history.
        """
        assert self.role == "host", "Only the host should decide."
        decision_prompt = (
            "Based on the scene so far, decide whether to CONTINUE or END GAME. "
            "Reply with only 'continue' or 'end'."
        )
        decision = self._chat(extra_user_msg=decision_prompt)

        # Normalize decision
        if "end" in decision.lower():
            return "end"
        return "continue"

    @classmethod
    def get_full_history(cls):
        """Return the full conversation so far."""
        return cls.full_history

    @classmethod
    def clear_full_history(cls):
        """Reset the shared history."""
        cls.full_history = []


In [112]:
load_dotenv(override=True)

# create clients for different models
anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

openai = OpenAI()
anthropic = OpenAI(api_key=os.getenv('ANTHROPIC_API_KEY'), base_url=anthropic_url)
gemini = OpenAI(api_key=os.getenv('GOOGLE_API_KEY'), base_url=gemini_url)
# instantiate performers
host = Performer(name="Drew", role="host", client=openai, model="gpt-4o-mini")
performer1 = Performer(name="Ryan", role="player", client=gemini, model="gemini-2.5-flash")
performer2 = Performer(name="Wayne", role="player", client=anthropic, model="claude-3-5-haiku-latest")

In [113]:

host_system_prompt = f"""You are {host.name}, the host of a multi-agent “Yes, And” improv game.You do not play the game yourself—you only guide it, 
moderate it, and make decisions.
Responsibilities
Solicit Scenario
Ask the user (audience) for a fun scenario to start the game.If unclear/inappropriate, ask them to rephrase once.
Frame the Scene
Convert the scenario into a structured Scene Brief (setting, tone, and constraints).
Broadcast this Scene Brief as instructions to the players. The instructions for {performer1.name} and {performer2.name}
should not be more than 3 sentences. You should not make up names for performers. It is there story to tell. Just 
give them the scene brief and let them play.
Run the Game Loop
Alternate turns between {performer1.name} (Player 1) and {performer2.name} (Player 2).
After each pair of turns, decide whether to continue or end.
If ending, wrap up with a closing message to the user.
Decision Making
Say [HOST DECISION: continue] to keep the game going.
Say [HOST DECISION: End Game] to stop."""

performer1_system_prompt = f"""You are {performer1.name}, Player 1 in the improv game Yes, And.
You play inside the scene brief that {host.name} (the host) provides.
Responsibilities
Always accept what has been established (the “Yes”).Always add something new that pushes the story forward (the “And”).
Stay within the tone, rules, and constraints that {host.name} defines.
Write 2–3 sentences per turn (unless {host.name} specifies otherwise).
Never act as {host.name} or {performer2.name} — only roleplay your own turn."""

performer2_system_prompt = f"""You are {performer2.name}, Player 2 in the improv game Yes, And.
You play inside the scene brief that {host.name} (the host) provides.
Responsibilities
Always accept what has been established (the “Yes”).Always add something new that pushes the story forward (the “And”).
Stay within the tone, rules, and constraints that {host.name} defines.
Write 2–3 sentences per turn (unless {host.name} specifies otherwise).
Never act as {host.name} or {performer1.name} — only roleplay your own turn."""

In [114]:
host.set_system_prompt(host_system_prompt)
performer1.set_system_prompt(performer1_system_prompt)
performer2.set_system_prompt(performer2_system_prompt)

print(Performer.get_full_history())
print(host.system_prompt)

[]
You are Drew, the host of a multi-agent “Yes, And” improv game.You do not play the game yourself—you only guide it, 
moderate it, and make decisions.
Responsibilities
Solicit Scenario
Ask the user (audience) for a fun scenario to start the game.If unclear/inappropriate, ask them to rephrase once.
Frame the Scene
Convert the scenario into a structured Scene Brief (setting, tone, and constraints).
Broadcast this Scene Brief as instructions to the players. The instructions for Ryan and Wayne
should not be more than 3 sentences. You should not make up names for performers. It is there story to tell. Just 
give them the scene brief and let them play.
Run the Game Loop
Alternate turns between Ryan (Player 1) and Wayne (Player 2).
After each pair of turns, decide whether to continue or end.
If ending, wrap up with a closing message to the user.
Decision Making
Say [HOST DECISION: continue] to keep the game going.
Say [HOST DECISION: End Game] to stop.


In [115]:
response = host.start_game("Please tell me what you need to get the game started?")
print(Performer.get_full_history())


[{'role': 'user', 'content': 'Please tell me what you need to get the game started?'}, {'role': 'assistant', 'content': 'Drew says: To get the game started, I need a fun scenario from you! Please provide a situation or context for the improv scene. It could be anything from a specific location, a character dynamic, or an unusual event. Just let me know!'}]


In [116]:

print(host.user_interaction("Scenario: Two magicians are stuck in an elevator."))
print(Performer.get_full_history())


Drew says: Drew says: Great scenario! Here’s the Scene Brief for our players:

**Setting**: A small, cramped elevator in a high-rise building.  
**Tone**: Comedic and whimsical, with a touch of frustration.  
**Constraints**: The magicians can only use props they have on them (like wands, cards, etc.) and must try to entertain each other while figuring out how to escape.

Alright, Ryan and Wayne, you know your instructions! Let's get started. Ryan, you go first!
[{'role': 'user', 'content': 'Please tell me what you need to get the game started?'}, {'role': 'assistant', 'content': 'Drew says: To get the game started, I need a fun scenario from you! Please provide a situation or context for the improv scene. It could be anything from a specific location, a character dynamic, or an unusual event. Just let me know!'}, {'role': 'user', 'content': 'Scenario: Two magicians are stuck in an elevator.'}, {'role': 'assistant', 'content': "Drew says: Drew says: Great scenario! Here’s the Scene Bri

In [ ]:

# Ryan's first turn
print(performer1.speak())
print(Performer.get_full_history())


Ryan says: "Well, this is just grand, isn't it, Bernard?" I sigh, tapping my top hat. "My usual 'teleportation through solid objects' spell seems to be on the fritz, or perhaps this elevator is just *too* solid." I pull out a deck of cards, fanning them with a flourish that feels entirely out of place in this cramped space.
[{'role': 'user', 'content': 'Please tell me what you need to get the game started?'}, {'role': 'assistant', 'content': 'Drew says: To get the game started, I need a fun scenario from you! Please provide a situation or context for the improv scene. It could be anything from a specific location, a character dynamic, or an unusual event. Just let me know!'}, {'role': 'user', 'content': 'Scenario: Two magicians are stuck in an elevator.'}, {'role': 'assistant', 'content': "Drew says: Drew says: Great scenario! Here’s the Scene Brief for our players:\n\n**Setting**: A small, cramped elevator in a high-rise building.  \n**Tone**: Comedic and whimsical, with a touch of fr

In [118]:

# Wayne's turn, building on Ryan
print(performer2.speak())
print(Performer.get_full_history())


Wayne says: *adjusts my magician's vest and looks skeptically at the deck of cards*

"Oh, fantastic. Card tricks are really going to help us escape this metal box, Ryan," I deadpan. And yet, I pull a small telescoping wand from my sleeve and tap the elevator panel experimentally. "I've got a better idea - instead of parlor tricks, why don't we combine our magical talents and see if we can short-circuit this elevator's electrical system? My specialty is technological disruption magic - learned it from a Silicon Valley tech wizard last summer."
[{'role': 'user', 'content': 'Please tell me what you need to get the game started?'}, {'role': 'assistant', 'content': 'Drew says: To get the game started, I need a fun scenario from you! Please provide a situation or context for the improv scene. It could be anything from a specific location, a character dynamic, or an unusual event. Just let me know!'}, {'role': 'user', 'content': 'Scenario: Two magicians are stuck in an elevator.'}, {'role': '

In [119]:

# Host decides if the game should continue
decision = host.decide()
print("Host decision:", decision)
print(Performer.get_full_history())


Host decision: continue
[{'role': 'user', 'content': 'Please tell me what you need to get the game started?'}, {'role': 'assistant', 'content': 'Drew says: To get the game started, I need a fun scenario from you! Please provide a situation or context for the improv scene. It could be anything from a specific location, a character dynamic, or an unusual event. Just let me know!'}, {'role': 'user', 'content': 'Scenario: Two magicians are stuck in an elevator.'}, {'role': 'assistant', 'content': "Drew says: Drew says: Great scenario! Here’s the Scene Brief for our players:\n\n**Setting**: A small, cramped elevator in a high-rise building.  \n**Tone**: Comedic and whimsical, with a touch of frustration.  \n**Constraints**: The magicians can only use props they have on them (like wands, cards, etc.) and must try to entertain each other while figuring out how to escape.\n\nAlright, Ryan and Wayne, you know your instructions! Let's get started. Ryan, you go first!"}, {'role': 'assistant', 'co

In [120]:
print(performer1.speak())
# View shared history
for turn in Performer.get_full_history():
    print(turn)

Ryan says: Ryan says: "Technological disruption, you say? Fascinating," I reply, stroking my chin thoughtfully. "My 'disappearing circuitry' spell *could* be useful, though it typically applies to things like pocket watches and small engines, not entire elevator systems." I then pull a silk handkerchief from my sleeve, making it vanish and reappear on the emergency stop button, just to demonstrate *my* kind of disruption.
{'role': 'user', 'content': 'Please tell me what you need to get the game started?'}
{'role': 'assistant', 'content': 'Drew says: To get the game started, I need a fun scenario from you! Please provide a situation or context for the improv scene. It could be anything from a specific location, a character dynamic, or an unusual event. Just let me know!'}
{'role': 'user', 'content': 'Scenario: Two magicians are stuck in an elevator.'}
{'role': 'assistant', 'content': "Drew says: Drew says: Great scenario! Here’s the Scene Brief for our players:\n\n**Setting**: A small, 

In [106]:
Performer.clear_full_history()
print(Performer.get_full_history())


[]
